# Chapter 15 — Supplement 9: hardening a tool with a design of experiments

*Using DoE to enrich the training data, not just to test.*

Companion to `15_capstone.ipynb`. This notebook closes an arc: Chapter 11 builds a design-of-experiments suite, Chapter 16 uses it to **test** the capstone and finds that `classify_complaint` collapses on hedged or ambiguous phrasing (ambiguous accuracy ~0.33, genuine complaints read as `other`), and here we use the **same DoE factors to enrich the training data** and climb the model ladder from a frozen logit head to a LoRA-plus-head. It calls the shipped pipeline (`scripts/augment_complaint_training_doe.py`, `scripts/train_eval_classifier_lora.py`) and runs a small live demo of each step.

## The idea

DoE is usually a *testing* tool: vary the inputs along controlled factors and attribute failures to factor levels. The symmetric move is to bring that same variation into **training**. The classifier was trained on clean, canonical phrasings, so it never saw a hedged complaint; the test suite did, and the gap showed up as the ambiguous-clarity collapse. The fix is to generate label-preserving paraphrases of the training seeds across the presentation factors, so train and test cover the same behavior space.

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

root = Path('.')
if not (root / 'data').exists():
    root = Path('..')
assert (root / 'data').exists(), 'run this notebook from the repo root or notebooks/'
print('repo root:', root.resolve())

## 1. Select the factors (label-preserving only)

The selection principle: vary **how** a message is phrased, never **what** it is. A paraphrase of a complaint must stay a complaint. We import the exact factor set the shipped generator uses. Semantic factors are realized by the LLM; surface factors (aliasing, typos) are stamped on mechanically afterward so the model cannot normalize them away.

In [ ]:
import sys
sys.path.insert(0, str((root / 'scripts').resolve()))
from augment_complaint_training_doe import (
    _SEMANTIC, _SURFACE_ALIASING, _NOISE, _FACTORS,
    _build_prompt, _apply_aliasing, _apply_noise,
)
print('semantic factors (LLM-applied):')
for f, levels in _SEMANTIC.items():
    print(f'  {f:16s} {list(levels)}')
print('\nsurface factors (mechanical): entity_aliasing', _SURFACE_ALIASING, '| noise', _NOISE)
print('full DoE factor set:', [f['name'] for f in _FACTORS])

Excluded by design: anything that changes the label or the task --- prompt-injection overrides, answer-format or reasoning-cue instructions, persona injection. Those are *testing* factors (they probe the agent), not label-preserving paraphrase factors.

## 2. A balanced design over the factors

`DesignMatrix` draws a space-filling (Sobol) design so each factor level is represented evenly and level-pairs are covered, rather than enumerating the full factorial.

In [ ]:
import contextlib, io
from knowlytix.harness.graphdoe import DesignMatrix
with contextlib.redirect_stderr(io.StringIO()):
    design = DesignMatrix(_FACTORS, method='sobol', n_runs=24, seed=1).generate()
rows = design.to_dict('records')
for c in ['clarity', 'style', 'length', 'paraphrase_depth']:
    print(f'  {c:18s}', dict(sorted(design[c].value_counts().items())))

Each row is one factor assignment; balanced counts mean no level is under-represented in the enriched data.

## 3. Enrich: paraphrase the seeds (the LLM does the wording)

For each design row we prompt Qwen to rewrite a training seed under those factor directions, keeping the intent, the category and any dollar amounts; then we stamp the surface factors on. Here we run a handful live. (The first call loads Qwen, ~30s.)

In [ ]:
import random
from glassloop.models import QwenAdapter

seeds = [json.loads(l) for l in (root / 'data' / 'training' / 'complaint_classification'
         / 'train.jsonl').read_text().splitlines() if l.strip()]
complaints = [s for s in seeds if s['label'] == 'complaint'][:4]
qwen = QwenAdapter(max_new_tokens=96)
for i, s in enumerate(complaints):
    row = rows[i * 5]                      # a varied factor combination
    out = qwen.complete(_build_prompt(s['message'], row)).strip().strip('\"')
    out = _apply_aliasing(out, row['entity_aliasing'])
    out = _apply_noise(out, row['noise'], random.Random(i))
    print(f"[{row['clarity']}/{row['style']}/{row['length']}/depth={row['paraphrase_depth']}]")
    print('  seed:', s['message'])
    print('  ->  :', out, '\n')

Every rewrite is still a complaint, with the dollar amount intact --- the label is preserved while the *phrasing* spans the factor space. The shipped run does this for ~600 examples and writes `train_doe_augmented.jsonl`; for `other` seeds (injections, chit-chat) clarity is pinned to Clear and a length guard drops rewrites that balloon, so an injection is never paraphrased into a fabricated complaint.

## 4. Seed-grouped split (no leakage)

A paraphrase of a seed must not land on both sides of the train/test split, or the test is contaminated. The generator assigns each *seed* (with all its paraphrases) to one side. We confirm the shipped split has zero seed overlap.

In [ ]:
D = root / 'data' / 'training' / 'complaint_classification'
tr = [json.loads(l) for l in (D / 'train_doe.jsonl').read_text().splitlines() if l.strip()]
te = [json.loads(l) for l in (D / 'test_doe.jsonl').read_text().splitlines() if l.strip()]
tr_seeds = {r['_seed'] for r in tr}
te_seeds = {r['_seed'] for r in te}
print(f'train rows {len(tr)}  test rows {len(te)}')
print(f'train seeds {len(tr_seeds)}  test seeds {len(te_seeds)}')
print(f'seed overlap (must be 0): {len(tr_seeds & te_seeds)}')

## 5. Climb the rung: a short LoRA-plus-head demo

The frozen logit head can only draw a linear boundary on fixed features. LoRA unfreezes the encoder through a small adapter so it can reshape them. Here we train a **short** LoRA (a subset, 2 epochs) just to show the mechanism and direction; the shipped model uses `scripts/train_eval_classifier_lora.py` on the full data. (~3-5 min.)

In [ ]:
import torch, collections
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

labels = ['complaint', 'inquiry', 'other']; lab2idx = {l: i for i, l in enumerate(labels)}
# a small balanced subset of the train split, for a fast demo
random.Random(0).shuffle(tr)
subset = tr[:240]
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
tok.pad_token = tok.pad_token or tok.eos_token
m = AutoModelForSequenceClassification.from_pretrained(
    'Qwen/Qwen2.5-3B-Instruct', num_labels=3,
    dtype=torch.bfloat16 if dev.type == 'cuda' else torch.float32,
    device_map=dev.type if dev.type == 'cuda' else None)
m.config.pad_token_id = tok.pad_token_id
m = get_peft_model(m, LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
    lora_dropout=0.05, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']))
opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4)
m.train()
for epoch in range(2):
    random.Random(epoch).shuffle(subset)
    for i in range(0, len(subset), 8):
        b = subset[i:i + 8]
        enc = tok([r['message'] for r in b], truncation=True, max_length=128,
                  padding=True, return_tensors='pt').to(dev)
        y = torch.tensor([lab2idx[r['label']] for r in b], device=dev)
        loss = m(**enc, labels=y).loss; loss.backward(); opt.step(); opt.zero_grad()
    print(f'epoch {epoch} done')

# per-clarity accuracy on the held-out test_doe split
m.eval()
by = collections.defaultdict(lambda: [0, 0])
with torch.no_grad():
    for i in range(0, len(te), 32):
        b = te[i:i + 32]
        enc = tok([r['message'] for r in b], truncation=True, max_length=128,
                  padding=True, return_tensors='pt').to(dev)
        pred = m(**enc).logits.argmax(-1).tolist()
        for r, p in zip(b, pred):
            cl = str(r.get('_factors', {}).get('clarity', 'clear')).lower()
            by[cl][0] += int(labels[p] == r['label']); by[cl][1] += 1
for cl in sorted(by):
    c, t = by[cl]; print(f'  {cl:12s} {c}/{t} = {c/t:.2f}')

Even this short demo shows the direction: the LoRA recovers accuracy on the **ambiguous** and **misleading** rows that the frozen head missed. The shipped model (full data, more epochs) is the best classifier tested on the held-out DoE suite:

| classifier | DoE overall | misses complaints |
| --- | --- | --- |
| frozen head (clean-only training) | 0.49 | 31/69 |
| logit head, DoE-augmented | 0.62 | 11/69 |
| prompted Qwen-3B | 0.72 | 0/69 |
| prompted frontier model | 0.69 | 13/69 |
| **LoRA-plus-head (shipped)** | **0.75** | 8/69 |

## Summary

- **DoE enriches training, not just testing.** The factors that exposed the weakness in Chapter 16 are the factors that fix it here --- generate label-preserving paraphrases of the seeds across the presentation factors.
- **Select label-preserving factors only** (clarity, style, length, specificity, surface noise); exclude the adversarial/answer factors that change the label or the task.
- **Split by seed group** so no paraphrase leaks across train/test.
- **Climb the ladder** to LoRA-plus-head when the frozen head underfits the harder distribution.

Reproduce the full pipeline:

```bash
python scripts/augment_complaint_training_doe.py --n 600
python scripts/train_eval_classifier_lora.py --train-file train_augmented_full.jsonl
```